# FYOE — train v4 model on Colab T4

**Before you start:** click `Runtime` → `Change runtime type` → set hardware accelerator to **T4 GPU**.

Then `Runtime` → `Run all`. ~15-20 min total. At the end you get `saved_model_v4.zip` to download.

**What changed from v3:**
v3 hit 99.06% on the IID test set but only **52.4% on the seed suite** (82 curated real-world messages). That 46.6 pp gap was the work item. v4 closes it by adding 8 banks of targeted training examples in `training/v4_failure_modes.py`:
- `V4_NEGATION` (220) — “I’m not paying for X”, “don’t transfer the X yet” (v3 baseline 25%)
- `V4_PAST_TENSE` (220) — “paid X yesterday”, “ubered home last night” (v3 43%)
- `V4_MULTI_INTENT` (480) — compound messages, must fire 2-3 intents (v3 0/6)
- `V4_CODE_MIXED` (400) — Hindi-English Roman script (v3 25%)
- `V4_CONTACT_FIX` (400) — every “call X” / “text X” / pronoun form (v3 had **0% recall** on contact)
- `V4_DONT_FORGET` (140) — negation marker but action SHOULD fire
- `V4_AMBIGUOUS_QUIET` (150) — “set it for 6”, “send 20” — must NOT fire (v3 fired alarm at 0.95)
- `V4_QUERY_NOT_ACTION` (120) — “did I pay rent” — info queries that look like commands

Total dataset goes from ~16K → ~19K. **Contact intent positives jumped 600 → 1254** (the 0% recall fix). Negatives went from ~5K → 6.5K (negation/past/ambiguous push).

**Goal:** seed-suite accuracy 52.4% → ≥90% with no regression on IID test (≥98%).

## 1. Setup — fresh VM, clone repo, install deps

**Before running:** push your v4 changes (`generate_data.py` import + integration block, `v4_failure_modes.py`) to a branch named `v4` on GitHub. The clone below pulls that branch.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!rm -rf paychat-model
# Clone v4 branch — fall back to v3 if v4 doesn't exist yet so the notebook still runs.
!git clone -b v4 https://github.com/Akash-Cheerla/paychat-model.git || git clone -b v3 https://github.com/Akash-Cheerla/paychat-model.git
%cd paychat-model
!git log -1 --oneline
!ls training/v4_failure_modes.py && echo 'v4 banks present' || echo 'WARNING: v4_failure_modes.py missing — push v4 branch first'

In [ ]:
# Pin transformers to last stable 4.x. transformers 5.0 has regressions in checkpoint loading.
!pip install -q 'transformers==4.46.3' 'tokenizers>=0.20,<0.21' sentencepiece scikit-learn
import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)

## 2. Generate training data (v4)

~19K examples: v3 positives + v4 failure-mode banks + per-intent hard negatives. The script also prints v4 category counts so you can verify the import worked before training.

In [ ]:
%cd /content/paychat-model/training
!python generate_data.py

In [ ]:
# Sanity check: confirm v4 categories actually landed in the dataset.
import json
from collections import Counter
ds = json.load(open('full_dataset.json'))
v4_cats = Counter(d['category'] for d in ds if d['category'].startswith('v4_'))
print(f'Total examples: {len(ds)}')
print(f'\nv4 categories present:')
for c, n in v4_cats.most_common():
    print(f'  {c:<22} {n}')
assert v4_cats, 'v4 categories missing — generate_data.py did not import V4_* banks'
n_contact = sum(1 for d in ds if d['labels']['contact'] == 1)
print(f'\ncontact positives: {n_contact}  (v3 had ~600; target ≥1000 to fix 0% recall)')

## 3. Fine-tune RoBERTa-base (v4)

Same hyperparameters as v3 (5 epochs, batch 32, lr 2e-5, max-len 128). Larger dataset → ~15-18 min on T4 instead of 10-15.

Healthy run: train loss drops ~0.3 → ~0.04 by epoch 5; val hamming reaches ~98%; per-intent thresholds re-tuned on val after the last epoch.

In [ ]:
!python train.py \
  --model roberta-base \
  --epochs 5 \
  --batch-size 32 \
  --max-len 128

## 4. Inspect results

In [ ]:
import json
from pathlib import Path
model_dir = Path('/content/paychat-model/saved_model')
print('Files in saved_model/:')
for f in sorted(model_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:35} {size_kb:>10.1f} KB')

print('\nLearned per-intent thresholds:')
with open(model_dir / 'thresholds.json') as f:
    thresholds = json.load(f)
for intent, thr in thresholds.items():
    print(f'  {intent:<14} {thr:.2f}')

print('\nPer-intent test metrics:')
with open(model_dir / 'training_report.json') as f:
    report = json.load(f)
print(f"  test exact-match: {report['test_exact_match']:.2%}")
print(f"  test hamming:     {report['test_hamming']:.2%}")
print()
print(f"  {'intent':<14} {'precision':>10} {'recall':>8} {'f1':>7}")
for intent, m in report['per_intent'].items():
    print(f"  {intent:<14} {m['precision']:>9.1%} {m['recall']:>7.1%} {m['f1']:>6.1%}")

## 5. Seed-suite regression (the real test)

v3 scored **43/82 (52.4%)** on this suite. v4 should reach **≥74/82 (≥90%)**. This is the number that actually matters — it’s the 82 messages that broke v3.

If v4 doesn't beat 80% here, do not ship. Look at the failure list, expand the relevant V4_* bank in `v4_failure_modes.py`, regenerate, retrain.

In [ ]:
%cd /content/paychat-model
!python eval/run_seed_baseline.py 2>&1 | tail -20

In [ ]:
# Pull the headline numbers from the report so they're easy to compare to v3.
import json
rpt = json.load(open('/content/paychat-model/eval/baseline_report.json'))
print(f"v4 seed-suite: {rpt['passed']}/{rpt['total']} passed ({rpt['passed']/rpt['total']*100:.1f}%)")
print(f"v4 IID test:   {rpt['test_exact_match']:.2f}% exact match")
print(f"v4 latency:    {rpt['ms_per_case']:.0f} ms/case")
print()
print('By tag:')
for tag, info in sorted(rpt['by_tag'].items(), key=lambda x: x[1]['passed']/x[1]['total']):
    rate = info['passed']/info['total']*100
    print(f"  {tag:<25} {info['passed']:>2}/{info['total']:<2}  ({rate:.0f}%)")

## 6. Sanity check — fire real messages through the trained model

In [ ]:
import torch, json
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = '/content/paychat-model/saved_model'
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval().cuda()
with open(f'{MODEL_DIR}/thresholds.json') as f:
    thresholds = json.load(f)
labels = list(mdl.config.id2label.values()) if mdl.config.id2label else list(thresholds.keys())

# Mix of v3-passing samples + the exact v3 failures we're targeting.
samples = [
    # v3 broke on these:
    'call dad',                                     # v3: fired task; v4: contact
    'text mom',                                     # v3: fired task; v4: contact
    'call him',                                     # v3: nothing; v4: contact (resolver asks who)
    "I'm not paying for pizza",                     # v3: fired money; v4: silent
    'paid rent already',                            # v3: fired bills; v4: silent
    'set it for 6',                                 # v3: alarm 0.95; v4: silent (or low)
    'send 20',                                      # v3: money 0.9; v4: silent (or low)
    'uber to JFK at 5am tomorrow',                  # v3: ride only; v4: ride+maps+alarm
    'yaar venmo me 20 for pizza',                   # code-mixed money
    'mom ko phone karna hai',                       # code-mixed contact
    "don't forget to call mom thursday",            # negation marker, action fires
    'how much did I spend on food this month',      # query, not action
    # v3 already passes these; verify no regression:
    'venmo me 20 bucks for pizza',
    'remind me to call mom tomorrow at 6pm',
    'flight to Tokyo next month',
    'I love Paris',                                 # must NOT fire travel
    'watched Stranger Things last week',            # must NOT fire video
]

for text in samples:
    enc = tok(text, return_tensors='pt', truncation=True, max_length=128).to('cuda')
    with torch.no_grad():
        logits = mdl(**enc).logits[0]
    probs = torch.sigmoid(logits).cpu().tolist()
    fired = [(labels[i], probs[i]) for i in range(len(labels)) if probs[i] >= thresholds.get(labels[i], 0.5)]
    fired.sort(key=lambda x: -x[1])
    fired_str = ', '.join(f'{l}={p:.2f}' for l, p in fired) or '(none)'
    print(f'{text!r:<60} -> {fired_str}')

## 7. Zip and download `saved_model/`

Drop the unzipped folder into your local repo at `./saved_model/`, then re-run `python eval/run_seed_baseline.py` locally to refresh `eval/baseline_report.md`. Compare to the v3 baseline you already have on disk.

In [ ]:
%cd /content/paychat-model
!zip -r saved_model_v4.zip saved_model/ -x '*.bin.tmp' '*.cache*' > /dev/null
!ls -lh saved_model_v4.zip
from google.colab import files
files.download('saved_model_v4.zip')